In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 07 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 07.C1 — Construct threshold edges

Return all index pairs whose Euclidean distance is <= epsilon. Do not include self-edges.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Enumerate i<j so every undirected pair occurs once.

</details>

In [3]:
def threshold_edges(X,epsilon):
    X=np.asarray(X,dtype=float)
    return {pair for pair in combinations(range(len(X)),2) if np.linalg.norm(X[pair[0]]-X[pair[1]])<=epsilon}

In [4]:
X=np.array([[0,0],[1,0],[1,1],[0,1]],float)
assert len(threshold_edges(X,1))==4
assert len(threshold_edges(X,np.sqrt(2)))==6
print("Square has four side edges at 1 and six edges at sqrt(2)")

Square has four side edges at 1 and six edges at sqrt(2)


**Why this works and what it does not establish:** This produces only the graph layer. Rips homology requires the higher simplices implied by its cliques.

## 07.C2 — Fill every clique triangle

Given a vertex count and a set of sorted edge tuples, return all triples whose three edges occur.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Test every two-vertex face of each candidate triple.

</details>

In [5]:
def clique_triangles(n,edges):
    return {triple for triple in combinations(range(n),3) if all(edge in edges for edge in combinations(triple,2))}

In [6]:
assert clique_triangles(3,{(0,1),(0,2),(1,2)})=={(0,1,2)}
assert clique_triangles(3,{(0,1),(1,2)})==set()
print("Pairwise edge criterion for Rips triangles checked")

Pairwise edge criterion for Rips triangles checked


**Why this works and what it does not establish:** The condition is correct for a Rips complex. It is not the common-intersection condition defining a Cech simplex.

## 07.C3 — Assign a simplex entrance time

For a nonempty simplex of point indices return its diameter; vertices enter at zero.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Take the maximum pairwise distance, not the mean.

</details>

In [7]:
def diameter_time(X,simplex):
    X=np.asarray(X,dtype=float)
    return max((float(np.linalg.norm(X[i]-X[j])) for i,j in combinations(simplex,2)),default=0.)

In [8]:
X=np.array([[0,0],[1,0],[1,1]],float)
assert diameter_time(X,(0,))==0
assert np.isclose(diameter_time(X,(0,1,2)),np.sqrt(2))
for s in closure([(0,1,2)]): assert diameter_time(X,s)<=np.sqrt(2)+1e-12
print("Face times never exceed coface diameter")

Face times never exceed coface diameter


**Why this works and what it does not establish:** Taking a subset cannot increase a set diameter, which gives the face-before-coface filtration property.

## 07.C4 — Show scale dependence

Return the ratio of a chosen pairwise distance after multiplying all coordinates by a positive scale factor to its original distance. Assume the two points differ.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Euclidean norms are positively homogeneous.

</details>

In [9]:
def distance_scale_ratio(X,i,j,factor):
    X=np.asarray(X,dtype=float)
    return float(np.linalg.norm(factor*X[i]-factor*X[j])/np.linalg.norm(X[i]-X[j]))

In [10]:
X=load_iris_data().iloc[:,1:5].to_numpy()
assert np.isclose(distance_scale_ratio(X,0,75,10),10)
print("Uniform rescaling multiplies edge-length filtration units by ten")

Uniform rescaling multiplies edge-length filtration units by ten


**Why this works and what it does not establish:** A uniform scale change preserves the ordinary topology of the underlying shape but changes persistence coordinates. Alpha squared-radius units would transform by the square of the factor.

## 07.C5 — Audit a real Rips filtration

On the first twelve measurement rows, build the supplied Rips filtration through triangles at cutoff 0.5. Return counts of vertices, edges, and triangles.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The cutoff and selected rows are part of the experiment specification.

</details>

In [11]:
def iris_rips_counts():
    X=load_iris_data().iloc[:12,1:5].to_numpy()
    f=rips_filtration(X,max_homology=1,max_edge=.5)
    return [sum(len(s)==k+1 for s,v in f) for k in range(3)]

In [12]:
counts=iris_rips_counts()
assert counts[0]==12 and all(c>=0 for c in counts)
print("First twelve Iris rows, cutoff 0.5; V,E,F:",counts)

First twelve Iris rows, cutoff 0.5; V,E,F: [12, 31, 35]


**Why this works and what it does not establish:** This intentionally tiny subset is a teaching example, not a representative population sample. State that limitation before interpreting its components or loops.